In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os
from pinecone import Pinecone

from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from pinecone import DenseVectorQuery

In [ ]:
from dotenv import load_dotenv

load_dotenv("../.env")

In [ ]:
# Extract Data From the PDF Files
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )

    documents = loader.load()

    return documents

extracted_data = load_pdf_file(data="../data/")

In [ ]:
# extracted_data

In [ ]:
# Split the Data into Text Chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [ ]:
text_chunks = text_split(extracted_data)

print("Length of Text Chunks:", len(text_chunks))

In [ ]:
# Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    return embeddings

In [ ]:
embeddings = download_hugging_face_embeddings()

query_result = embeddings.embed_query("Hello world")

print("Length:", len(query_result))

In [ ]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")

if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY is missing")

# Initialize Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"

# Get existing indexes
existing_indexes = [idx.name for idx in pc.indexes.list()]

# Create index if it doesn't already exist
if index_name not in existing_indexes:
    pc.indexes.create(
        name=index_name,
        schema={
            "fields": {
                "vector": {
                    "type": "dense_vector",
                    "dimension": 384,
                    "metric": "cosine"
                }
            }
        },
        deployment={
            "deployment_type": "managed",
            "cloud": "aws",
            "region": "us-east-1"
        }
    )

    print("Pinecone Index created successfully!")
else:
    print("Pinecone Index already exists!")

# Verify Index
print("Available Indexes:", [idx.name for idx in pc.indexes.list()])


In [ ]:
import hashlib

# Connect to existing Pinecone index
index = pc.index(index_name)

# Namespace for our medical documents
namespace = "medical-rag"

# Validate extracted data
if not text_chunks:
    raise ValueError("No text chunks found!")

# Upload chunks in batches
batch_size = 32
total_uploaded = 0

for start in range(0, len(text_chunks), batch_size):

    batch = text_chunks[start:start + batch_size]

    # Extract text
    texts = [doc.page_content for doc in batch]

    # Generate embeddings locally
    vectors = embeddings.embed_documents(texts)

    documents = []

    for doc, vector in zip(batch, vectors):

        # Stable ID prevents duplicate records
        content_key = (
            str(doc.metadata.get("source", "")) +
            str(doc.metadata.get("page", "")) +
            doc.page_content
        )

        doc_id = hashlib.sha256(
            content_key.encode("utf-8")
        ).hexdigest()

        documents.append({
            "_id": doc_id,
            "vector": vector,
            "text": doc.page_content,
            "source": str(doc.metadata.get("source", "")),
            "page": int(doc.metadata.get("page", 0))
        })

    # Store documents in Pinecone
    response = index.documents.upsert(
        namespace=namespace,
        documents=documents
    )

    total_uploaded += response.upserted_count

    print(
        f"Uploaded {total_uploaded}/{len(text_chunks)} chunks"
    )

print("Medical Chunks uploaded successfully!")


Test Semantic Search


In [ ]:
from pinecone import DenseVectorQuery

# Test medical question
query = "What are the symptoms of diabetes?"

# Convert question into a vector
query_vector = embeddings.embed_query(query)

# Search Pinecone
results = index.documents.search(
    namespace="medical-rag",
    top_k=3,
    score_by=[
        DenseVectorQuery(
            field="vector",
            values=query_vector
        )
    ],
    include_fields=["text", "source", "page"]
)

# Display results
for i, match in enumerate(results.matches, start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", match.score)
    print("Text:", match.get("text"))
    print("Source:", match.get("source"))
    print("Page:", match.get("page"))

In [ ]:
query = "What are the common signs and symptoms of diabetes mellitus?"

query_vector = embeddings.embed_query(query)

results = index.documents.search(
    namespace="medical-rag",
    top_k=5,
    score_by=[
        DenseVectorQuery(
            field="vector",
            values=query_vector
        )
    ],
    include_fields=["text", "source", "page"]
)

for i, match in enumerate(results.matches, start=1):
    print(f"\n--- Result {i} ---")
    print("Score:", round(match.score, 4))
    print("Text:", match.get("text"))
    print("Page:", match.get("page"))

In [ ]:
# Initialize local LLM using Ollama
llm = ChatOllama(
    model="qwen3:4b-instruct",
    temperature=0.4,
    num_predict=500
)

In [ ]:
# Retrieve relevant medical documents from Pinecone
def retrieve_documents(inputs):

    question = inputs["input"]

    query_vector = embeddings.embed_query(question)

    results = index.documents.search(
        namespace="medical-rag",
        top_k=3,
        score_by=[
            DenseVectorQuery(
                field="vector",
                values=query_vector
            )
        ],
        include_fields=["text", "source", "page"]
    )

    documents = []

    for match in results.matches:
        documents.append(
            Document(
                page_content=match.get("text") or "",
                metadata={
                    "source": match.get("source"),
                    "page": match.get("page"),
                    "score": match.score
                }
            )
        )

    return documents

In [ ]:
# Create Retriever
retriever = RunnableLambda(retrieve_documents)

In [ ]:
# Define system prompt
system_prompt = (
    "You are a medical information assistant for "
    "question-answering tasks. "
    "Use only the following retrieved context to answer "
    "the question. "
    "If the context does not contain enough information, "
    "say that you don't know based on the provided documents. "
    "Do not invent medical facts or diagnoses. "
    "Use three sentences maximum and keep the answer concise. "
    "Your answers are for educational purposes only, "
    "not personalized medical advice."
    "\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

In [ ]:
# Create Question Answer Chain
question_answer_chain = create_stuff_documents_chain(llm,prompt)

In [ ]:
# Create RAG Chain
rag_chain = create_retrieval_chain(
    retriever,
    question_answer_chain
)

In [ ]:
response = rag_chain.invoke({
    "input": "What is Acromegaly and gigantism?"
})

print("Answer:")
print(response["answer"])

In [ ]:
response = rag_chain.invoke({
    "input": "What is stats?"
})

print("Answer:")
print(response["answer"])

# Baseline Evaluation


In [ ]:
# Medical RAG - Baseline Evaluation Questions
evaluation_questions = [
    "What is acromegaly?",
    "What is the difference between acromegaly and gigantism?",
    "What are the symptoms of diabetes mellitus?",
    "What causes hypertension?",
    "What are the symptoms of anemia?",
    "What is asthma?",
    "How is hypothyroidism diagnosed?",
    "What is pneumonia?",
    "What are the symptoms of migraine?",
    "What is machine learning?"
]

print("Total Evaluation Questions:", len(evaluation_questions))

In [ ]:
# Baseline Retrieval Evaluation

baseline_results = []

for question in evaluation_questions:

    query_vector = embeddings.embed_query(question)

    results = index.documents.search(
        namespace="medical-rag",
        top_k=3,
        score_by=[
            DenseVectorQuery(
                field="vector",
                values=query_vector
            )
        ],
        include_fields=["text", "source", "page"]
    )

    retrieved_docs = [
        {
            "score": float(match.score),
            "text": match.get("text") or "",
            "source": match.get("source"),
            "page": match.get("page")
        }
        for match in results.matches
    ]

    baseline_results.append({
        "question": question,
        "documents": retrieved_docs
    })

    print(f"\nQuestion: {question}")

    for i, doc in enumerate(retrieved_docs, 1):
        print(
            f"  Result {i}: "
            f"Score={doc['score']:.4f}, "
            f"Page={doc['page']}"
        )

In [ ]:
# Display retrieved chunks for manual relevance evaluation
for result in baseline_results:

    print("=" * 80)
    print("QUESTION:", result["question"])
    print("=" * 80)

    for i, doc in enumerate(result["documents"], start=1):

        print(f"\n--- Chunk {i} ---")
        print("Score:", round(doc["score"], 4))
        print("Page:", doc["page"])
        print("Text:")
        print(doc["text"])
        print()

    print("\n")

In [ ]:
from sentence_transformers import CrossEncoder

# Load Reranking Model
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)

print("Reranking Model loaded successfully!")

In [ ]:
# Test question
query = "What are the symptoms of diabetes mellitus?"

# Convert question into embedding
query_vector = embeddings.embed_query(query)

# Retrieve 10 candidate documents
results = index.documents.search(
    namespace="medical-rag",
    top_k=10,
    score_by=[
        DenseVectorQuery(
            field="vector",
            values=query_vector
        )
    ],
    include_fields=["text", "source", "page"]
)

candidate_docs = [
    {
        "text": match.get("text") or "",
        "source": match.get("source"),
        "page": match.get("page"),
        "similarity_score": float(match.score)
    }
    for match in results.matches
]

print("Retrieved Candidates:", len(candidate_docs))

In [ ]:
# Prepare question-document pairs
pairs = [
    (query, doc["text"])
    for doc in candidate_docs
]

# Calculate reranking scores
rerank_scores = reranker.predict(pairs)

# Attach reranking scores
for doc, score in zip(candidate_docs, rerank_scores):
    doc["rerank_score"] = float(score)

# Sort by reranking score
reranked_docs = sorted(
    candidate_docs,
    key=lambda doc: doc["rerank_score"],
    reverse=True
)

# Select top 3 results
top_reranked_docs = reranked_docs[:3]

for i, doc in enumerate(top_reranked_docs, start=1):
    print(f"\n--- Reranked Result {i} ---")
    print("Rerank Score:", round(doc["rerank_score"], 4))
    print("Original Similarity:", round(doc["similarity_score"], 4))
    print("Page:", doc["page"])
    print("Text:", doc["text"])

In [ ]:
print("===== BASELINE: TOP 3 =====")

for i, doc in enumerate(candidate_docs[:3], start=1):
    print(f"{i}. Page {doc['page']} | "
          f"Similarity: {doc['similarity_score']:.4f}")


print("\n===== RERANKED: TOP 3 =====")

for i, doc in enumerate(top_reranked_docs, start=1):
    print(f"{i}. Page {doc['page']} | "
          f"Rerank Score: {doc['rerank_score']:.4f}")

In [ ]:
# Compare the third result before and after reranking

print("===== BASELINE - RESULT 3 =====")
print("Page:", candidate_docs[2]["page"])
print("Text:")
print(candidate_docs[2]["text"])

print("\n" + "=" * 60 + "\n")

print("===== RERANKED - RESULT 3 =====")
print("Page:", top_reranked_docs[2]["page"])
print("Text:")
print(top_reranked_docs[2]["text"])

In [ ]:
# Improved Chunking Experiment
improved_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

improved_chunks = improved_splitter.split_documents(extracted_data)

print("Original Chunks:", len(text_chunks))
print("Improved Chunks:", len(improved_chunks))

In [ ]:
# Inspect improved chunks for problematic medical topics
keywords = [
    "diabetes mellitus",
    "hypothyroidism"
]

for keyword in keywords:

    matching_chunks = [
        doc for doc in improved_chunks
        if keyword.lower() in doc.page_content.lower()
    ]

    print("=" * 80)
    print("KEYWORD:", keyword)
    print("Matching Chunks:", len(matching_chunks))

    for i, doc in enumerate(matching_chunks[:3], start=1):
        print(f"\n--- Chunk {i} ---")
        print("Page:", doc.metadata.get("page"))
        print("Length:", len(doc.page_content))
        print("Text:")
        print(doc.page_content)

remove
Book page headers.
Printed book page numbers.
PDF production lines, such as GEM - 0001 to 0432.


In [ ]:
import re


def clean_medical_text(text):

    # Remove PDF production lines
    text = re.sub(
        r"(?m)^GEM\s*-\s*\d+\s+to\s+\d+.*$",
        "",
        text
    )

    # Remove encyclopedia page headers
    text = re.sub(
        r"(?m)^GALE ENCYCLOPEDIA OF MEDICINE\s*\d*\s+\d+.*$",
        "",
        text
    )

    # Normalize extra spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize excessive empty lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
cleaned_documents = []

for doc in extracted_data:

    cleaned_text = clean_medical_text(
        doc.page_content
    )

    if cleaned_text:
        cleaned_documents.append(
            Document(
                page_content=cleaned_text,
                metadata=doc.metadata.copy()
            )
        )

print("Original Documents:", len(extracted_data))
print("Cleaned Documents:", len(cleaned_documents))

In [ ]:
cleaned_chunks = improved_splitter.split_documents(
    cleaned_documents
)

print("V1 Chunks:", len(text_chunks))
print("V2 Chunks:", len(improved_chunks))
print("Cleaned V2 Chunks:", len(cleaned_chunks))

In [ ]:
# Document Cleaning Validation
original_chars = sum(
    len(doc.page_content)
    for doc in extracted_data
)

cleaned_chars = sum(
    len(doc.page_content)
    for doc in cleaned_documents
)

print("Original Characters:", original_chars)
print("Cleaned Characters:", cleaned_chars)

print(
    "Characters Removed:",
    original_chars - cleaned_chars
)

print(
    "Reduction Percentage:",
    round(
        (original_chars - cleaned_chars)
        / original_chars * 100,
        2
    ),
    "%"
)

# Find pages that became empty
empty_pages = []

for doc in extracted_data:

    cleaned_text = clean_medical_text(
        doc.page_content
    )

    if not cleaned_text:
        empty_pages.append(doc.metadata)

print("\nEmpty Pages After Cleaning:", len(empty_pages))
print("Empty Page Metadata:", empty_pages)

In [ ]:
# Inspect the page removed during cleaning

first_page = extracted_data[0]

print("Page:", first_page.metadata.get("page"))
print("Original Text Length:", len(first_page.page_content))
print("Original Text:")
print(repr(first_page.page_content))

In [ ]:
# Inspect cleaned chunks

for i, doc in enumerate(cleaned_chunks[:5], start=1):
    print("=" * 70)
    print(f"Chunk {i}")
    print("Page:", doc.metadata.get("page"))
    print("Length:", len(doc.page_content))
    print("Text:")
    print(doc.page_content[:500])
    print()

Cover Pages

Copyright & Editorial Information

Table of Contents

Preface

Actual Medical Articles


In [ ]:
# Inspect the first 30 pages after cleaning

for doc in cleaned_documents[:30]:

    page = doc.metadata.get("page")
    text = doc.page_content.strip()

    print("=" * 60)
    print(f"PDF Page: {page}")
    print(f"Text Length: {len(text)}")
    print("Preview:")
    print(text[:250].replace("\n", " "))
    print()

In [ ]:
# Remove introductory and non-medical pages
medical_documents = [
    doc for doc in cleaned_documents
    if doc.metadata.get("page", -1) >= 14
]

print("Cleaned Documents:", len(cleaned_documents))
print("Medical Documents:", len(medical_documents))
print(
    "Removed Introductory Pages:",
    len(cleaned_documents) - len(medical_documents)
)

In [ ]:
# Create V2 chunks from medical content only
medical_chunks_v2 = improved_splitter.split_documents(
    medical_documents
)

print("Original V1 Chunks:", len(text_chunks))
print("Cleaned V2 Chunks:", len(cleaned_chunks))
print("Medical V2 Chunks:", len(medical_chunks_v2))

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a local vector database for V2 evaluation
vectorstore_v2 = FAISS.from_documents(
    documents=medical_chunks_v2,
    embedding=embeddings
)

print("FAISS V2 Vector Store created successfully!")
print("Indexed Chunks:", len(medical_chunks_v2))

In [ ]:
# Test V2 Semantic Retrieval
test_questions = [
    "What are the symptoms of diabetes mellitus?",
    "How is hypothyroidism diagnosed?"
]

for question in test_questions:

    results_v2 = vectorstore_v2.similarity_search(
        query=question,
        k=3
    )

    print("\n" + "=" * 70)
    print("QUESTION:", question)

    for i, doc in enumerate(results_v2, start=1):
        print(f"\n--- V2 Result {i} ---")
        print("Page:", doc.metadata.get("page"))
        print("Text:", doc.page_content)

In [ ]:
# Check medical document coverage

search_terms = [
    "hypothyroidism",
    "diabetes mellitus"
]

for term in search_terms:

    matching_pages = [
        doc for doc in medical_documents
        if term.lower() in doc.page_content.lower()
    ]

    print("=" * 65)
    print("Search Term:", term)
    print("Matching Pages:", len(matching_pages))
    print(
        "Page Numbers:",
        [doc.metadata.get("page") for doc in matching_pages]
    )

In [ ]:
# Inspect all pages mentioning hypothyroidism
term = "hypothyroidism"

for doc in medical_documents:

    text = doc.page_content

    if term.lower() in text.lower():

        page = doc.metadata.get("page")

        print("=" * 80)
        print("PDF PAGE:", page)
        print("=" * 80)

        # Find each occurrence of the term
        import re

        for match in re.finditer(
            re.escape(term),
            text,
            flags=re.IGNORECASE
        ):

            start = max(0, match.start() - 350)
            end = min(len(text), match.end() + 500)

            print("\n--- CONTEXT ---")
            print(text[start:end])